# 第 3 周练习 — Stayez 合成推荐数据工作室（Synthetic Data Studio）

**学生：** Vagz1216  
**课程：** LLM 工程 — Andela AI 工程训练营  
**练习：** 第 3 周 — 合成数据生成器

---

## 练习目标（理念）

构建 **Stayez Synthetic Data Studio**：面向肯尼亚高端旅宿预订平台 Stayez，为个性化推荐 / RAG / ML 实验准备**可下载的结构化合成数据**。

本工具通过多提供商调用开源与云端模型（Hugging Face 上的 Llama 3.2、Groq 上的 Llama 3.3、以及 Gemini），生成贴近肯尼亚语境的 JSON 记录，再转成 Pandas / CSV。

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Hugging Face Inference | `InferenceClient` + `chat.completions.create` |
| OpenAI 兼容客户端 | Groq / Gemini 的 `base_url` |
| System / User Prompt | 强制「只输出 JSON 数组」 |
| Gradio Blocks | 选数据集类型、条数、引擎并预览 / 下载 |

## 功能要点

- 三类数据集：**Guest Profiles**、**Property Catalog**、**Booking & Review History**（键名保持英文，与代码一致）
- 输出严格 JSON → DataFrame → CSV  
- 交互式 Gradio UI；需在 `.env` 配置 `GROQ_API_KEY` / `GEMINI_API_KEY` / `HF_TOKEN`

## 怎么跑

1. 安装依赖：`openai`、`gradio`、`pandas`、`python-dotenv`、`huggingface_hub`  
2. 配置 `.env` 后从上到下运行；最后一格 `demo.launch` 打开界面  


In [ ]:
# ========== 单元格 1：导入依赖 ==========

# os：读环境变量里的 API Key
import os
# json：把 schema / 模型输出在 Python 与字符串之间转换
import json
# pandas：把 JSON 列表变成表格，并导出 CSV
import pandas as pd
# gradio：搭交互式 Blocks UI
import gradio as gr
# load_dotenv：从 .env 加载密钥，避免写死在笔记本里
from dotenv import load_dotenv
# OpenAI 客户端：同时用于 Groq / Gemini 的 OpenAI 兼容端点
from openai import OpenAI
# Hugging Face 推理客户端：Week 3 核心——远程跑开源 Instruct 模型
from huggingface_hub import InferenceClient


In [ ]:
# ========== 单元格 2：API 密钥与多提供商客户端 ==========

# override=True：.env 中的值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 从环境变量读取 Groq Key（变量名须与 .env 键一致）
groq_api_key   = os.getenv('GROQ_API_KEY')
# Gemini Key（走 Google OpenAI 兼容端点）
gemini_api_key = os.getenv('GEMINI_API_KEY')
# Hugging Face Token：Inference API / 门控模型鉴权
hf_token       = os.getenv('HF_TOKEN')

# 构造 HF 推理客户端（第 3 周核心）：后续 chat.completions 走 Hub
hf_client = InferenceClient(token=hf_token)

# Groq 客户端：OpenAI SDK + Groq 的 OpenAI 兼容 base_url（通常生成更快）
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# Gemini 客户端：同一套 OpenAI SDK，换 Google 的兼容 base_url
gemini = OpenAI(api_key=gemini_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# HF 侧开源 Instruct 模型 ID（字符串必须与 Hub 一致）
MODEL_HF     = "meta-llama/Llama-3.2-3B-Instruct"
# Groq 侧大模型 ID
MODEL_GROQ   = "llama-3.3-70b-versatile"
# Gemini 侧轻量 flash 模型 ID
MODEL_GEMINI = "gemini-1.5-flash"

# 就绪提示（英文 print 保持原样，便于对照运行输出）
print("✅ API Clients initialized (Hugging Face, Groq, Gemini). Ready to generate data!")


In [ ]:
# ========== 单元格 3：三类推荐相关数据集的 schema 字典 ==========
# 键名 / description / fields 英文会原样拼进 user_prompt，供 LLM 遵守——勿翻译改写

DATASET_SCHEMAS = {
    # --- 数据集 A：宾客画像（推荐冷启动常用）---
    "Guest Profiles": {
        # 给模型看的业务语境说明
        "description": "Demographic & preference profiles for Stayez guests.",
        # 每个字段：名字 → 约束说明（会 json.dumps 进 prompt）
        "fields": {
            "guest_id": "Unique string ID usually starting with GUEST_",
            "age": "Integer between 18 and 65",
            "travel_style": "String (e.g., Backpacker, Luxury, Business, Family, Solo)",
            "budget_per_night_ksh": "Integer (e.g., 2500 to 25000)",
            "preferred_activities": "List of strings (e.g., [Hiking, Nightlife, Beach, Quiet Retreat])"
        }
    },
    # --- 数据集 B：肯尼亚语境房源目录 ---
    "Property Catalog": {
        "description": "A list of realistic Kenyan Airbnb-style properties.",
        "fields": {
            "property_id": "Unique string ID starting with PROP_",
            "name": "Catchy name for the property (e.g., 'Cozy Kilimani Studio')",
            "city": "String (e.g., Nairobi, Mombasa, Nakuru, Kisumu, Diani, Naivasha)",
            "type": "String (e.g., Villa, Studio, 1BR, 2BR, Cabin)",
            "price_ksh": "Integer (pricing matching the type)",
            "vibe_tags": "List of strings (e.g., [Romantic, Workspace, Scenic, Central])"
        }
    },
    # --- 数据集 C：预订 + 点评历史（可训排序/情感）---
    "Booking & Review History": {
        "description": "Logs of past stays with ratings and text reviews. Needs to be realistic.",
        "fields": {
            "booking_id": "Unique string ID starting with BKG_",
            "guest_id": "String ID of the guest",
            "property_id": "String ID of the property",
            "nights_stayed": "Integer between 1 and 14",
            "rating": "Integer 1 to 5",
            "review_text": "A 1-2 sentence review justifying the rating. Make it sound like a real Kenyan or international guest."
        }
    }
}


In [ ]:
# ========== 单元格 4：合成数据生成器（多模型分支 + JSON 清洗） ==========

def generate_synthetic_data(dataset_type, num_records, model_choice):
    # 用下拉选中的数据集名取出 description / fields
    schema_info = DATASET_SCHEMAS[dataset_type]
    
    # system：角色设定 + 禁止 markdown 围栏（英文 prompt 契约，勿改）
    system_prompt = """
    You are a masterful synthetic data generation engine for Stayez, a Kenyan travel tech company.
    Your task is to generate highly realistic, varied, and structured mock data for our recommendation systems.
    You must return ONLY a raw JSON array format matching the requested schema.
    DO NOT wrap the JSON in markdown blocks like ```json ... ```. Just return the raw array [ { ... } ].
    DO NOT include any commentary.
    """
    
    # user：条数、类型、字段 schema；强调肯尼亚语境（f-string 原文勿改）
    user_prompt = f"""
    Generate exactly {num_records} records of type: {dataset_type}.
    Context: {schema_info['description']}
    
    The fields for each JSON object MUST be exactly:
    {json.dumps(schema_info['fields'], indent=2)}
    
    Make the data incredibly realistic to the Kenyan context (real city names, realistic KSh prices, relatable reviews).
    Output only the JSON array.
    """

    try:
        # ---- 分支：Hugging Face Inference（开源 Llama 3.2 3B）----
        if model_choice == "Hugging Face (Llama 3.2 3B)":
            # 远程 chat.completions；messages 含 system + user
            response = hf_client.chat.completions.create(
                model=MODEL_HF,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.7,
                max_tokens=4000
            )
            # 取第一条回复文本并去首尾空白
            raw_output = response.choices[0].message.content.strip()
            
        else:
            # ---- 分支：Groq 或 Gemini（下拉文案必须与 if 条件全等）----
            # 非 Groq 时落到 gemini 客户端
            client = groq if model_choice == "Groq (Llama 3 70B)" else gemini
            # 同步选择对应模型 ID 常量
            model = MODEL_GROQ if model_choice == "Groq (Llama 3 70B)" else MODEL_GEMINI
            
            # OpenAI 兼容 create：参数形状与 HF 分支一致
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.7,
                max_tokens=4000
            )
            raw_output = response.choices[0].message.content.strip()
        
        # ---- 后处理：去掉模型偶发的 ```json 围栏 ----
        if raw_output.startswith("```json"):
            raw_output = raw_output[7:]
        if raw_output.startswith("```"):
            raw_output = raw_output[3:]
        if raw_output.endswith("```"):
            raw_output = raw_output[:-3]
            
        # 再 strip 一次，准备 json.loads
        raw_output = raw_output.strip()
        
        # 字符串 → Python list[dict]
        data_list = json.loads(raw_output)
        
        # 表格化，便于 Gradio Dataframe 预览
        df = pd.DataFrame(data_list)
        
        # 写临时 CSV；空格变下划线，路径模式保持原样供 File 组件下载
        csv_path = f"/tmp/stayez_{dataset_type.lower().replace(' ', '_')}.csv"
        df.to_csv(csv_path, index=False)
        
        # 成功三元组：表、路径、状态文案（状态字符串保持原样）
        return df, csv_path, "✅ Generation Successful!"
        
    except json.JSONDecodeError:
        # JSON 解析失败：空表 + 截断原文（错误文案勿改，便于对照）
        return pd.DataFrame(), None, f" Error: The model did not return valid JSON.\n\nRaw output:\n{raw_output[:500]}"
    except Exception as e:
        # 网络 / 鉴权 / 其它运行时错误
        return pd.DataFrame(), None, f"Unexpected Error: {str(e)}"


In [ ]:
# ========== 单元格 5：Gradio Blocks UI ==========

# Soft 绿色主题；demo 是整页应用对象
with gr.Blocks(theme=gr.themes.Soft(primary_hue="green")) as demo:
    # 标题与说明（展示给用户的英文 UI 文案保持原样）
    gr.Markdown("## 🧪 Stayez Recommendation System - Synthetic Data Studio")
    gr.Markdown("Generate incredibly realistic, structured datasets using Llama 3 via Hugging Face/Groq to train Stayez ML models.")
    
    with gr.Row():
        # 左栏：配置控件
        with gr.Column(scale=1):
            # 数据集类型：选项来自 DATASET_SCHEMAS 的键
            dataset_dd = gr.Dropdown(
                choices=list(DATASET_SCHEMAS.keys()), 
                value="Guest Profiles", 
                label="Dataset Objective"
            )
            # 生成条数滑块：5–50，步长 5
            num_records_sl = gr.Slider(
                minimum=5, 
                maximum=50, 
                value=10, 
                step=5, 
                label="Number of Records"
            )
            # 引擎下拉：字符串必须与 generate_synthetic_data 内分支判断一致
            model_dd = gr.Dropdown(
                choices=["Hugging Face (Llama 3.2 3B)", "Groq (Llama 3 70B)", "Gemini 1.5 Flash"], 
                value="Hugging Face (Llama 3.2 3B)", 
                label="Engine"
            )
            # 主按钮：触发生成
            generate_btn = gr.Button("Generate Data ⚡", variant="primary")
            # 状态文本（只读）
            status_txt = gr.Textbox(label="Status", interactive=False)
            # CSV 下载位
            download_btn = gr.File(label="Download CSV", interactive=False)
            
        # 右栏：表格预览（更宽）
        with gr.Column(scale=2):
            output_df = gr.Dataframe(label="Data Preview", wrap=True)

    # 绑定点击：inputs 顺序对应函数参数；outputs 对应返回三元组
    generate_btn.click(
        fn=generate_synthetic_data,
        inputs=[dataset_dd, num_records_sl, model_dd],
        outputs=[output_df, download_btn, status_txt]
    )

# 启动 Gradio；inbrowser=True 尝试自动打开浏览器
demo.launch(inbrowser=True)
